# BioCLIP zero-shot classification on the iNaturalist 2021 validation split

Replaces the 2024 notebook `BioClip-INAT21-Fixed (2).ipynb` (archived verbatim under
`archive/2024-original/notebooks/`). It scores every image of the official iNat21 `val` split
(100,000 images, 10 per species, 10,000 species) against one text prompt per species and reports
top-1 accuracy.

**Historical recorded outputs (2024, not reproduced here):**

| checkpoint | prompt | top-1 | source |
|---|---|---|---|
| `imageomics/bioclip` (TreeOfLife-10M) | `{taxonomy} with common name {common}` | 0.6398 | archived notebook, cell 21 |
| `imageomics/bioclip-vit-b-16-inat-only` | same | 0.7195 | archived notebook, cell 25 |

Known issues in how those numbers were produced (see `docs/changes.md`): text embeddings were
**not** L2-normalised (image embeddings were), the second checkpoint was run with the first
checkpoint's preprocessing object, and labels relied on prediction index == category id.
This notebook normalises both sides, loads preprocessing from each checkpoint, and maps
labels explicitly, so a rerun is a *corrected* measurement, not a reproduction of the 2024 values.

Validity note: TreeOfLife-10M contains the iNat21 **training** split, and the `inat-only`
checkpoint was trained on iNat21, so this is zero-shot in the prompt sense only: all 10,000
species were seen during pre-training; the `val` images were not (see `docs/evaluation_validity.md`).


> **Smoke test vs. real run.** When the dataset is not present under `BIODIV_DATA_ROOT`
> (or `BIODIV_SMOKE_TEST=1` is set) this notebook runs on a tiny synthetic fixture with a
> weight-free stand-in encoder. That exercises the code path only; the numbers it prints are
> **not** results. A real run needs the dataset and downloads the pinned BioCLIP checkpoint
> (~600 MB per model) from the Hugging Face Hub.

In [ ]:
import json, os, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO)]
os.environ.setdefault("BIODIV_DATA_ROOT", str(REPO / "data"))      # where inat_comp/<year>/ lives
os.environ.setdefault("BIODIV_OUTPUT_DIR", str(REPO / "outputs"))

from biodiv.paths import get_paths
paths = get_paths()  # BIODIV_DATA_ROOT / BIODIV_CACHE_DIR / BIODIV_OUTPUT_DIR
print("data_root =", paths.data_root.resolve())

In [ ]:
VAL_JSON = paths.data_root / "inat_comp/2021/val.json"
IMAGE_ROOT = paths.data_root / "inat_comp/2021"
SMOKE = os.environ.get("BIODIV_SMOKE_TEST", "auto")
if SMOKE == "auto":
    SMOKE = "0" if VAL_JSON.exists() else "1"
SMOKE = SMOKE == "1"
LIMIT = int(os.environ.get("BIODIV_LIMIT", "0")) or None  # evaluate only the first N images (quick checks)
print("mode:", "SMOKE TEST (synthetic fixture, no weights)" if SMOKE else "REAL DATA")

In [ ]:
from biodiv.clip_eval import evaluate_zero_shot, duplicate_prompt_groups
from biodiv.species_data import inat_split

if SMOKE:
    from biodiv.fixtures import load_fixture_rare_species
    records, table = load_fixture_rare_species()
else:
    records, table = inat_split(VAL_JSON, IMAGE_ROOT)
if LIMIT:
    records = records[:LIMIT]

PROMPT_STYLE, PREFIX = "taxonomic_common", ""   # the 2024 prompt format
prompts = table.prompts(PROMPT_STYLE, PREFIX)
print(f"{len(table)} species, {len(records)} image records, {len(set(prompts))} unique prompts")
print("example prompt:", prompts[0])
print("duplicate prompt groups:", duplicate_prompt_groups(prompts) or "none")

In [ ]:
from biodiv.encoders import KNOWN_CHECKPOINTS, OpenClipEncoder

CHECKPOINTS = {
    "bioclip": ("imageomics/bioclip", KNOWN_CHECKPOINTS["imageomics/bioclip"]),
    "bioclip-inat21-only": ("imageomics/bioclip-vit-b-16-inat-only", KNOWN_CHECKPOINTS["imageomics/bioclip-vit-b-16-inat-only"]),
}

def load_encoder(name):
    if SMOKE:
        from biodiv.fixtures import fixture_encoder
        return fixture_encoder()
    repo_id, revision = CHECKPOINTS[name]
    return OpenClipEncoder(repo_id, revision=revision, cache_dir=str(paths.cache_dir))

In [ ]:
results = {}
for name in CHECKPOINTS:
    encoder = load_encoder(name)
    results[name] = evaluate_zero_shot(encoder, records, table, PROMPT_STYLE, PREFIX, batch_size=32, num_workers=0)
    r = results[name]
    print(f"{name:22s} top-1 (ties->first) {r['top1_accuracy_first']:.4f} | top-1 (ties->any) {r['top1_accuracy_any']:.4f} "
          f"| top-5 {r['top5_accuracy']:.4f} | evaluated {r['n_evaluated']} skipped {r['n_skipped']} | {r['seconds']:.0f}s")

In [ ]:
out = paths.output_dir / ("smoke_bioclip_inat21_val.json" if SMOKE else "bioclip_inat21_val.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"smoke_test": SMOKE, "limit": LIMIT, "results": results}, indent=2))
print("wrote", out)

## Reading the output

* `top1_accuracy_first` is plain argmax accuracy. `top1_accuracy_any` counts an image as correct
  when its true species ties for the best score; on iNat21 every prompt is unique, so the two
  agree unless two categories produce byte-identical prompt text.
* `n_skipped` lists images that could not be decoded; they are excluded from the denominator and
  reported rather than silently shifting labels.
* A full run on CPU is slow (100,000 images); the 2024 runs took about 17-20 minutes on an A40.
